In [57]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import time
import io
from PIL import Image
import hashlib
import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

In [60]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

In [61]:
# This is the path I use
# DRIVER_PATH = '.../Desktop/Scraping/chromedriver 2'
# Put the path for your ChromeDriver here
options = Options()
options.add_experimental_option('excludeSwitches', ['enable-logging'])
options.add_argument("--ignore-certificate-error")
options.add_argument("--ignore-ssl-errors")
# DRIVER_PATH = 'C:/Users/Ronan Revadker/Documents/Dissertation Project/Notebooks/chromedriver.exe'
DRIVER_PATH = 'Notebooks/chromedriver.exe'
wd = webdriver.Chrome(executable_path=DRIVER_PATH, options=options)

In [62]:
def fetch_image_urls(query:str, max_links_to_fetch:int, wd:webdriver, sleep_between_interactions:int=1):
    def scroll_to_end(wd):
        wd.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(sleep_between_interactions)    
    
    # build the google query
    search_url = "https://www.google.com/search?safe=off&site=&tbm=isch&source=hp&q={q}&oq={q}&gs_l=img"

    # load the page
    wd.get(search_url.format(q=query))

    image_urls = set()
    image_count = 0
    results_start = 0
    while image_count < max_links_to_fetch:
        scroll_to_end(wd)

        # get all image thumbnail results
        thumbnail_results = wd.find_elements_by_css_selector("img.Q4LuWd")
        number_results = len(thumbnail_results)
        
        print(f"Found: {number_results} search results. Extracting links from {results_start}:{number_results}")
        
        for img in thumbnail_results[results_start:number_results]:
            # try to click every thumbnail such that we can get the real image behind it
            try:
                img.click()
                time.sleep(sleep_between_interactions)
            except Exception:
                continue

            # extract image urls    
            actual_images = wd.find_elements_by_css_selector('img.n3VNCb')
            for actual_image in actual_images:
                if actual_image.get_attribute('src') and 'http' in actual_image.get_attribute('src'):
                    image_urls.add(actual_image.get_attribute('src'))

            image_count = len(image_urls)

            if len(image_urls) >= max_links_to_fetch:
                print(f"Found: {len(image_urls)} image links, done!")
                break
        else:
            print("Found:", len(image_urls), "image links, looking for more ...")
            time.sleep(30)
            return
            load_more_button = wd.find_element_by_css_selector(".mye4qd")
            if load_more_button:
                wd.execute_script("document.querySelector('.mye4qd').click();")

        # move the result startpoint further down
        results_start = len(thumbnail_results)

    return image_urls

In [63]:
def persist_image(folder_path:str,url:str):
    try:
        image_content = requests.get(url).content

    except Exception as e:
        print(f"ERROR - Could not download {url} - {e}")

    try:
        image_file = io.BytesIO(image_content)
        image = Image.open(image_file).convert('RGB')
        file_path = os.path.join(folder_path,hashlib.sha1(image_content).hexdigest()[:10] + '.jpg')
        with open(file_path, 'wb') as f:
            image.save(f, "JPEG", quality=95)
        print(f"SUCCESS - saved {url} - as {file_path}")
    except Exception as e:
        print(f"ERROR - Could not save {url} - {e}")

In [64]:
def search_and_download(search_term:str,driver_path:str,target_path='dataset/scenario2/images',number_images=15):
    target_folder = os.path.join(target_path,'_'.join(search_term.lower().split(' ')))
    
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)

    with webdriver.Chrome(executable_path=driver_path) as wd:
        res = fetch_image_urls(search_term, number_images, wd=wd, sleep_between_interactions=0.5)
        
    url_list=res
    print(url_list)
    
    for elem in res:
        persist_image(target_folder,elem)
            
    return url_list

In [13]:
dataset = pd.read_csv('../dataset/text/train_json.csv')


In [10]:
def add_tuple_to_file(images_list):
    with open('../dataset/text/image_urls.csv', 'a') as images_file:
        images_file.writelines('\n'.join('{}; {}\n'.format(tup[0],tup[1]) for tup in images_list))

In [ ]:
new_li = [
# 'big fish', 
# 'bull market', # done
'chain reaction',
'elbow grease', # search images with links
# 'elbow room', # done
# 'fine line', # done
# 'front runner', # done
'insider trading' # done, but search images with links
]

data_new=dataset[dataset['compound'].isin(new_li)]
data_new

In [ ]:
# new_data = dataset.iloc[265:]
# new_data

In [ ]:
images_urls = []
count=0

dataset_len = len(dataset) - 1

for index, row in data_new.iterrows():
    
    compound = row['compound']
    print('Searching images for: ', compound)
    url_list = search_and_download(search_term=compound,
                   driver_path=DRIVER_PATH)
    images_urls.append((compound, url_list))
    print('add urls to file')
    add_tuple_to_file(images_urls)
    images_urls=[]
#     if not (index+1) % 5:
#         print('add urls to file')
#         add_tuple_to_file(images_urls)
#         images_urls=[]

In [ ]:
add_tuple_to_file(images_urls)